# 03 - Feature Engineering: RFM

This notebook turns the clean transactions from notebook 02 into one row per customer, with three numbers: Recency (how recently the customer bought), Frequency (how often) and Monetary (how much they spent). These numbers are what I use to segment the customers later.

## 1. Load the data

I load the clean sales and the cancellations that notebook 02 saved. A CSV file does not store column types, so I set them while reading instead of fixing them afterwards.

In [1]:
import pandas as pd

PROCESSED_DIR = "../data/processed"
sales = pd.read_csv(f"{PROCESSED_DIR}/sales_clean.csv", index_col="raw_index", parse_dates=["InvoiceDate"], 
                    dtype={"InvoiceNo": "string", "StockCode": "string", "CustomerID": "Int64"})
cancellations = pd.read_csv(f"{PROCESSED_DIR}/cancellations.csv", index_col="raw_index", parse_dates=["InvoiceDate"], 
                    dtype={"InvoiceNo": "string", "StockCode": "string", "CustomerID": "Int64"})
print(sales.shape)
print(cancellations.shape)
assert len(sales) == 522500, "sales row count does not match notebook 02"
assert len(cancellations) == 9286, "cancellations row count does not match notebook 02"


(522500, 8)
(9286, 8)


In [2]:
print("sales")
print(sales.dtypes)
print("cancellations")
print(cancellations.dtypes)

sales
InvoiceNo              string
StockCode              string
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID              Int64
Country                   str
dtype: object
cancellations
InvoiceNo              string
StockCode              string
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID              Int64
Country                   str
dtype: object


`sales` has 522,500 rows and `cancellations` has 9286 rows, the same as in notebook 02. `CustomerID` is read as `Int64`, so the missing customers stay empty instead of turning the column into a float.

## 2. Keep only rows with a customer and add Revenue

RFM is calculated per customer, so rows without a `CustomerID` can't be used. I leave `sales` and `cancellations` as they are and build two new tables, `rfm_sales` and `rfm_cancellations`, with only the rows that have a customer. I also add a `Revenue` column (`Quantity` times `UnitPrice`), which is the base for the Monetary value.

In [3]:
print(sales["CustomerID"].isnull().sum())
print(cancellations["CustomerID"].isnull().sum())

131354
383


In [4]:
has_customer_sales = sales["CustomerID"].notna()
rfm_sales = sales[has_customer_sales].copy()
has_customer_cancellations = cancellations["CustomerID"].notna()
rfm_cancellations = cancellations[has_customer_cancellations].copy()
print(rfm_sales.shape)
print(rfm_cancellations.shape)

assert rfm_sales['CustomerID'].isna().sum() == 0, "rfm_sales still contains missing CustomerID values"
assert rfm_cancellations['CustomerID'].isna().sum() == 0, "rfm_cancellations still contains missing CustomerID values"


(391146, 8)
(8903, 8)


In [5]:
rfm_sales["Revenue"] = rfm_sales["UnitPrice"] * rfm_sales["Quantity"]
rfm_cancellations["Revenue"] = rfm_cancellations["UnitPrice"] * rfm_cancellations["Quantity"]

assert (rfm_sales['Revenue'] > 0).all(), "Not all Revenue values in rfm_sales are greater than 0"
assert (rfm_cancellations['Revenue'] < 0).all(), "Not all Revenue values in rfm_cancellations are less than 0"

print("Total Revenue rfm_sales:", round(rfm_sales['Revenue'].sum(), 2))
print("Total Revenue rfm_cancellations:", round(rfm_cancellations['Revenue'].sum(), 2))

Total Revenue rfm_sales: 8491022.59
Total Revenue rfm_cancellations: -365688.89


In [6]:
print(rfm_sales["CustomerID"].nunique())
print(rfm_sales["InvoiceNo"].nunique())
print(rfm_cancellations["CustomerID"].nunique())

total_sales = rfm_sales['Revenue'].sum()
total_cancellations = abs(rfm_cancellations['Revenue'].sum())
cancellation_pct = (total_cancellations / total_sales) * 100
print(cancellation_pct)

4333
18400
1587
4.3067708997815775


131,354 rows in `sales` and 383 rows in `cancellations` had no customer and were left out. `rfm_sales` has 391,146 rows from 4,333 customers and 18,400 invoices, and its revenue adds up to 8,491,022.59. `rfm_cancellations` has 8,903 rows from 1,587 customers. Its revenue is negative (-365,688.89 in total, about 4.3% of the sales revenue), because a cancellation has a negative `Quantity` and a positive `UnitPrice`.

## 3. Choose the reference date

Recency is the number of days since a customer's last purchase, so I need a "today" to count from. The data stops on 9 December 2011, so using the real current date would make every Recency more than a thousand days too large. I take the reference date from the data itself.

In [7]:
first_date = rfm_sales["InvoiceDate"].min()
last_date = rfm_sales["InvoiceDate"].max()
difference_date = (last_date - first_date).days
print(first_date)
print(last_date)
print(difference_date)

2010-12-01 08:26:00
2011-12-09 12:50:00
373


In [8]:
reference_date = last_date.normalize() + pd.Timedelta(days=1)

assert reference_date > last_date, "reference_date must be strictly greater than last_date"

print(reference_date)

2011-12-10 00:00:00


The sales run from 2010-12-01 to 2011-12-09, which is 373 days. I use 2011-12-10 as the reference date, one day after the last transaction, so a customer who bought on the last day has a Recency of 1 instead of 0. Because the data ends on 9 December, the last month is incomplete, which means customers who looked inactive for a few days may simply be cut off by the end of the data.

## 4. Calculate Recency, Frequency and Monetary

For every customer I calculate three numbers from `rfm_sales`:

- Recency: the number of days between the reference date and the customer's last purchase. A smaller number means a more recent customer.
- Frequency: the number of different invoices. I count invoices, not rows, because one invoice has many product lines.
- Monetary: the total `Revenue` the customer spent.

For now Monetary only uses sales. I decide what to do with the cancellations in the next step.

In [9]:
rfm = rfm_sales.groupby("CustomerID").agg(
    last_purchase=("InvoiceDate", "max"),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum") 
)
rfm.shape

(4333, 3)

In [10]:
rfm["Recency"] = (reference_date - rfm["last_purchase"].dt.normalize()).dt.days

In [11]:
print(rfm.head())
print(rfm[["Recency", "Frequency", "Monetary"]].describe().round(2))

                 last_purchase  Frequency  Monetary  Recency
CustomerID                                                  
12347      2011-12-07 15:52:00          7   4310.00        3
12348      2011-09-25 13:13:00          4   1437.24       76
12349      2011-11-21 09:51:00          1   1457.55       19
12350      2011-02-02 16:01:00          1    294.40      311
12352      2011-11-03 14:37:00          7   1385.74       37
       Recency  Frequency   Monetary
count  4333.00    4333.00    4333.00
mean     93.22       4.25    1959.62
std     100.13       7.64    8460.57
min       1.00       1.00       2.90
25%      18.00       1.00     303.97
50%      51.00       2.00     661.32
75%     143.00       5.00    1628.56
max     374.00     206.00  279138.02


In [12]:
assert len(rfm) == rfm_sales["CustomerID"].nunique(), "Row count of rfm does not match unique customers in rfm_sales"
assert rfm.index.is_unique, "rfm index is not unique"
assert rfm[["Recency", "Frequency", "Monetary"]].isna().sum().sum() == 0, "Missing values found in RFM columns"
assert rfm["Recency"].min() == 1, "Minimum Recency is not 1"
assert (rfm["Frequency"] >= 1).all(), "Some Frequency values are less than 1"
assert (rfm["Monetary"] > 0).all(), "Some Monetary values are not greater than 0"
assert round(rfm["Monetary"].sum(), 2) == round(rfm_sales["Revenue"].sum(), 2), "Total Monetary does not match total Revenue"
assert rfm["Frequency"].sum() == rfm_sales["InvoiceNo"].nunique(), "Total Frequency does not match unique invoices"
print("All RFM checks passed")

All RFM checks passed


The `rfm` table has 4,333 rows, one per customer. The median customer bought 51 days before the reference date, placed 2 orders and spent 661.32. The mean Monetary (1,959.62) is much higher than the median, and the biggest customer spent 279,138.02 in 72 orders, so a few very large customers pull the average up. The checks confirm that the total Monetary matches the total revenue in `rfm_sales` and that the total Frequency matches the number of invoices.

## 5. Cancellations and Monetary

So far Monetary only counts sales. A cancelled order should reduce what a customer really spent, so I look at the cancellations per customer. There are three options: ignore them, subtract all of them, or subtract only the cancellations of real products. Some cancellations have non-product codes (`M`, `POST`, `D`, ...). Those codes were already left out of `sales`, so subtracting them would take away money that I never counted. I subtract only the product cancellations. I don't match each cancellation to its original invoice, so Frequency still counts every invoice.

In [13]:
is_product_code = rfm_cancellations["StockCode"].str.match(r"^\d{5}")
non_product_cancellations = rfm_cancellations[~is_product_code]
print(len(non_product_cancellations))
print(non_product_cancellations["Revenue"].sum())
print((non_product_cancellations.groupby("StockCode")["Revenue"].sum()).sort_values(ascending=True).head(4))


366
-136938.76
StockCode
M      -112165.39
POST    -11093.72
CRUK     -7933.43
D        -5696.22
Name: Revenue, dtype: float64


In [14]:
product_cancellations = rfm_cancellations[is_product_code].copy()
print(product_cancellations.shape)
print(product_cancellations["Revenue"].sum())

(8537, 9)
-228750.13


In [15]:
cancelled_by_customer = product_cancellations.groupby("CustomerID")["Revenue"].sum()
customer_not_in_rfm = set(cancelled_by_customer.index) - set(rfm.index)
print(len(cancelled_by_customer))
print(len(customer_not_in_rfm))

1532
28


In [16]:
rfm["Cancelled"] = cancelled_by_customer.reindex(rfm.index).fillna(0)
rfm["Monetary_net"] = (rfm["Monetary"] + rfm["Cancelled"]).round(2)
customer_has_cancellations = (rfm["Cancelled"] < 0).sum()

print(customer_has_cancellations)
print(round(rfm["Cancelled"].sum(), 2))
print(round(rfm["Monetary_net"].sum(), 2))

assert round(rfm["Monetary"].sum() + rfm["Cancelled"].sum(), 2) == round(rfm["Monetary_net"].sum(), 2), "Monetary_net does not add up"

1504
-224827.15
8266195.44


In [17]:
zero_or_negative_rfm = (
    rfm[rfm["Monetary_net"] <= 0][["Frequency", "Monetary", "Cancelled", "Monetary_net"]]
    .sort_values(by="Monetary_net", ascending=True)
)

print(len(zero_or_negative_rfm))
print((rfm["Monetary_net"] < 0).sum())
print(zero_or_negative_rfm)

13
2
            Frequency  Monetary  Cancelled  Monetary_net
CustomerID                                              
17548               1    103.30    -244.78       -141.48
16546               2    787.15    -883.08        -95.93
12454               1   3528.34   -3528.34          0.00
12558               1    269.96    -269.96          0.00
13672               2    186.39    -186.39          0.00
13762               1    306.72    -306.72          0.00
12607               1   1579.51   -1579.51          0.00
13364               1    134.96    -134.96          0.00
14792               1      6.20      -6.20          0.00
14557               1    788.38    -788.38          0.00
16878               1     13.30     -13.30          0.00
18268               1     25.50     -25.50          0.00
18274               1    175.92    -175.92          0.00


366 cancellation rows have a non-product code and are worth -136,938.76, which is about 37% of all cancelled value, mostly `M` (manual entries). I leave them out. The product cancellations cover 1,504 customers in `rfm` and take 224,827.15 off Monetary, so the total goes from 8,491,022.59 to 8,266,195.44. 28 other customers only have cancellations and no sales in the data, so they are not part of `rfm`. 13 customers end up with a net Monetary of zero or less (2 of them negative), because they cancelled all or almost all of what they bought. Two of them first looked slightly positive (about 0.0000000000005) because of floating point, so I round `Monetary_net` to 2 decimals. A Monetary of zero or less can't be used later (a log transform needs a positive number), so I decide what to do with them in the next step.

## 6. Final RFM table

Customers with a net Monetary of zero or less returned everything they bought, so there is nothing to say about their value, and a zero or negative number would break the log transform I plan to use before clustering. They are only 13 of 4,333 customers (0.3%), so I drop them and keep their IDs in a list. The final table has three columns, Recency, Frequency and Monetary, where Monetary is the net value after product cancellations.

In [18]:
non_positive_customers = (rfm.index[rfm["Monetary_net"] <= 0])
print(len(non_positive_customers))
print(non_positive_customers.tolist())

13
[12454, 12558, 12607, 13364, 13672, 13762, 14557, 14792, 16546, 16878, 17548, 18268, 18274]


In [19]:
rfm_final = (
    rfm.loc[rfm["Monetary_net"] > 0, ["Recency", "Frequency", "Monetary_net"]]
    .rename(columns={"Monetary_net": "Monetary"})
    .copy()
)
print(rfm_final.shape)
print(rfm_final.dtypes)

(4320, 3)
Recency        int64
Frequency      int64
Monetary     float64
dtype: object


In [20]:
assert len(rfm_final) == len(rfm) - len(non_positive_customers), "Row count of rfm_final does not match expected count"
assert rfm_final.index.is_unique, "rfm_final index is not unique"
assert list(rfm_final.columns) == ["Recency", "Frequency", "Monetary"], "rfm_final columns do not match expected names"
assert rfm_final.isna().sum().sum() == 0, "Missing values found in rfm_final"
assert rfm_final["Recency"].min() >= 1, "Minimum Recency in rfm_final is less than 1"
assert rfm_final["Frequency"].min() >= 1, "Minimum Frequency in rfm_final is less than 1"
assert (rfm_final["Monetary"] > 0).all(), "Some Monetary values in rfm_final are not greater than 0"
assert not rfm_final.index.isin(non_positive_customers).any(), "Some non_positive_customers are still present in rfm_final"

In [21]:
print(rfm_final.describe().round(2))
print(rfm_final.skew().round(2))

       Recency  Frequency   Monetary
count  4320.00    4320.00    4320.00
mean     93.12       4.26    1913.53
std     100.12       7.65    8299.49
min       1.00       1.00       2.90
25%      18.00       1.00     299.04
50%      51.00       2.00     650.42
75%     143.00       5.00    1610.46
max     374.00     206.00  278778.02
Recency       1.24
Frequency    11.94
Monetary     21.57
dtype: float64


In [22]:
rfm_final.to_csv(f"{PROCESSED_DIR}/rfm.csv", index=True)
check_rfm = pd.read_csv(f"{PROCESSED_DIR}/rfm.csv", index_col="CustomerID")

assert check_rfm.shape == rfm_final.shape, "Shape mismatch between saved and original DataFrame"
assert (check_rfm.index == rfm_final.index).all(), "CustomerID index mismatch between saved and original DataFrame"
assert (check_rfm.round(2).to_numpy() == rfm_final.round(2).to_numpy()).all(), "Table values mismatch after rounding to 2 decimal places"

The final `rfm` table has 4,320 customers and three columns, and I dropped the 13 customers with a net Monetary of zero or less. The median customer bought 51 days before the reference date, placed 2 orders and spent 650.42, while the mean Monetary is 1,913.53. The distributions are very skewed (skewness: Recency 1.24, Frequency 11.94, Monetary 21.57), so I will probably use a log transform before K-Means. The table is saved as `data/processed/rfm.csv`.

## 7. Summary

The result of this notebook is one table, `data/processed/rfm.csv`, with one row per customer:

| Column | Meaning |
|---|---|
| `Recency` | Days between the reference date (2011-12-10) and the customer's last purchase |
| `Frequency` | Number of different invoices |
| `Monetary` | Total spent on products, after subtracting the cancellations of real products |

How the table was built:

| Step | Result |
|---|---|
| Clean sales (`sales_clean.csv`) | 522,500 rows |
| Rows with a `CustomerID` | 391,146 rows from 4,333 customers |
| Customers with a net Monetary of zero or less (dropped) | 13 |
| Final `rfm` table | 4,320 customers |

Things to remember:

- The reference date is 2011-12-10, one day after the last transaction, so the most recent customers have a Recency of 1.
- Only cancellations of real products are subtracted from Monetary. Non-product codes (`M`, `POST`, `CRUK`, `D`) are ignored, like they are in `sales`.
- Frequency counts every invoice, because I don't match cancellations to their original invoices.
- 28 customers that only have cancellations are not in the table.
- All three columns are strongly skewed to the right (the skewness of Monetary is 21.57), so I will probably use a log transform before clustering.

Next is `04_eda.ipynb`, where I explore the RFM table and the sales trends with charts.